In [0]:
# =============================================================================
# Smart ERP – Sales Silver Layer Simulation  (Databricks / Delta Lake)
# Catalog  : smart_odoo
# Targets  :
#     silver.sale_order       (+2500 new rows per run)
#     silver.sale_order_line  (+up to 7500 new lines per run)
#
# HOW BATCHING WORKS
# ──────────────────
# PKs are derived from MAX() queries on the actual silver tables.
# Dropping tables resets counters automatically — no stale state.
# sim_run_state is kept only as a run counter log (used for RNG seed).
#
# FIXES vs previous version
# ──────────────────────────
# FIX 1 – PK block derived from MAX(sale_order_id) / MAX(sale_order_line_id)
#          instead of sim_run_state arithmetic.  Dropping tables and re-running
#          now correctly starts from row 1 every time.
# FIX 2 – invoice_status for cancel orders is always "no".
#          Previously could randomly return "invoiced" which is illogical.
#
# IDEMPOTENCY GUARANTEES
# ──────────────────────
# ✅ PKs read from MAX() on actual tables → survives DROP TABLE + re-run
# ✅ Each run gets a non-overlapping PK block → zero duplicate rows ever
# ✅ INSERT-APPEND only — PKs never collide, no MERGE needed
# ✅ State table updated AFTER successful write → safe on failure/retry
# ✅ Pre-flight uses NAMED dict fields → immune to column-index shifts
# ✅ Temp views dropped after use
#
# Business rules
# ──────────────
#   order_id       → MAX(sale_order_id)+1 … +BATCH_SIZE, globally unique
#   partner_id     → 51–250  (customers only)
#   product_id     → 1–100
#   price_unit     → list_price from silver.product_template
#   standard_price → always < list_price  (enforced in product layer)
#   tax_rate       → 14 %  (Egypt VAT)
#   order_state    → draft(10%) sent(15%) sale(45%) done(15%) cancel(15%)
#   invoice_status → derived from order_state (see _invoice_status below)
# =============================================================================

# ── 0. Imports ────────────────────────────────────────────────────────────────
import random
from datetime import datetime, timedelta, timezone

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    DoubleType, TimestampType,
)

spark = SparkSession.builder.getOrCreate()

# ── 1. Constants ──────────────────────────────────────────────────────────────
CATALOG         = "smart_odoo"
SCHEMA          = "silver"
SOURCE_DB       = "python_ingest"
SEED_BASE       = 42
BATCH_SIZE      = 2500
MAX_LINES_BATCH = BATCH_SIZE * 3        # 7 500 — maximum lines per batch
TAX_RATE        = 0.14

START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 5, 20)

COMPANIES  = [(1, "Alpha Corp"), (2, "Beta Ltd"), (3, "Gamma Inc")]
SALE_TEAMS = [(1, "Direct Sales"), (2, "Online Sales"), (3, "Field Sales")]
CURRENCIES = [(1, "EGP")]

# ── 2. Load reference data from silver ───────────────────────────────────────
spark.sql(f"USE CATALOG {CATALOG}")

products_map = {
    r.pid: (float(r.list_price), r.name)
    for r in spark.sql("""
        SELECT product_tmpl_id AS pid, list_price, name
        FROM   silver.product_template
    """).collect()
}
assert len(products_map) > 0, "❌ No products found in silver.product_template"

partners_map = {
    r.partner_id: r.partner_name
    for r in spark.sql("""
        SELECT partner_id, partner_name
        FROM   silver.res_partner
        WHERE  customer_rank > 0
    """).collect()
}

users_map = {
    r.user_id: r.user_name
    for r in spark.sql("""
        SELECT user_id, partner_name AS user_name
        FROM   silver.res_users
    """).collect()
}

# ── 3. Determine next PK block from actual tables  [FIX 1] ───────────────────
#
# We query MAX() directly from the target tables instead of computing
# (run_number - 1) * BATCH_SIZE from sim_run_state.
#
# Why this is better:
#   • Dropping a table and re-running now correctly restarts from 1.
#   • A partial failed write is automatically skipped (the MAX reflects
#     only committed rows, so the next run picks up exactly where the
#     last successful write left off).

ORDER_TABLE = f"{CATALOG}.{SCHEMA}.sale_order"
LINE_TABLE  = f"{CATALOG}.{SCHEMA}.sale_order_line"

def _max_id(table: str, col: str) -> int:
    """Return MAX(col) from table, or 0 if table does not exist yet."""
    try:
        row = spark.sql(f"SELECT COALESCE(MAX({col}), 0) AS m FROM {table}").collect()
        return int(row[0].m)
    except Exception:
        return 0   # table doesn't exist yet

max_order_id = _max_id(ORDER_TABLE, "sale_order_id")
max_line_id  = _max_id(LINE_TABLE,  "sale_order_line_id")

start_order_id = max_order_id + 1
end_order_id   = start_order_id + BATCH_SIZE
start_line_id  = max_line_id + 1
line_id        = start_line_id

# sim_run_state is kept as a run-counter log only (used for RNG seed stability)
STATE_TABLE = f"{CATALOG}.{SCHEMA}.sim_run_state"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
        table_name  STRING,
        run_number  INT,
        updated_at  TIMESTAMP
    ) USING DELTA
""")

state_row  = spark.sql(f"""
    SELECT run_number FROM {STATE_TABLE}
    WHERE  table_name = 'sale_orders'
""").collect()

run_number = (state_row[0].run_number + 1) if state_row else 1

print(f"\n{'='*60}")
print(f"  Run #{run_number}  |  "
      f"order_ids {start_order_id} → {end_order_id - 1}  |  "
      f"batch size {BATCH_SIZE}")
print(f"{'='*60}\n")

# ── 4. Helpers ────────────────────────────────────────────────────────────────
def _rdate(rng):
    return START_DATE + timedelta(
        days=rng.randint(0, (END_DATE - START_DATE).days)
    )

def _order_state(rng):
    r = rng.random()
    if   r < 0.5: return "draft"
    elif r < 0.20: return "sent"
    elif r < 0.30: return "sale"
    elif r < 0.70: return "done"
    else:          return "cancel"

def _invoice_status(state, rng):
    # [FIX 2] cancel → always "no".  Previously the else branch could
    # return rng.choice([...]) which included "invoiced" — illogical for
    # a cancelled order.  Valid values: "to invoice", "invoiced", "no".
    if   state in ("draft", "sent"): return "to invoice"
    elif state == "sale":            return rng.choice(["to invoice", "invoiced"])
    elif state == "done":            return "invoiced"
    else:                            return "no"   # cancel

# ── 5. Schemas ────────────────────────────────────────────────────────────────
ORDER_COLS = [
    "sale_order_id","partner_id","partner_name",
    "team_id","team_name","company_id","company_name",
    "user_id","user_name","currency_id","currency_name",
    "date_order","order_state",
    "amount_untaxed","amount_tax","amount_total",
    "pricelist_id","origin","client_order_ref",
    "commitment_date","validity_date",
    "created_at","updated_at","dwh_loaded_at","dwh_source_db",
]

LINE_COLS = [
    "sale_order_line_id","sale_order_id","sale_order_name",
    "company_id","company_name","currency_id","currency_name",
    "partner_id","partner_name","salesman_id","salesman_name",
    "product_id","product_name","product_uom_id","product_uom_name",
    "order_state","display_type","virtual_id","linked_virtual_id",
    "qty_delivered_method","invoice_status",
    "analytic_distribution","extra_tax_data","line_name",
    "product_uom_qty","price_unit","discount",
    "price_subtotal","price_total",
    "price_reduce_taxexcl","price_reduce_taxinc",
    "qty_delivered","qty_invoiced","qty_to_invoice",
    "untaxed_amount_invoiced","untaxed_amount_to_invoice",
    "is_downpayment","is_expense","collapse_prices","collapse_composition",
    "created_at","updated_at",
    "technical_price_unit","price_tax","customer_lead","is_optional",
    "dwh_loaded_at","dwh_source_db",
]

order_schema = StructType([
    StructField("sale_order_id",      IntegerType(),  False),
    StructField("partner_id",         IntegerType(),  True),
    StructField("partner_name",       StringType(),   True),
    StructField("team_id",            IntegerType(),  True),
    StructField("team_name",          StringType(),   True),
    StructField("company_id",         IntegerType(),  True),
    StructField("company_name",       StringType(),   True),
    StructField("user_id",            IntegerType(),  True),
    StructField("user_name",          StringType(),   True),
    StructField("currency_id",        IntegerType(),  True),
    StructField("currency_name",      StringType(),   True),
    StructField("date_order",         TimestampType(), True),
    StructField("order_state",        StringType(),   True),
    StructField("amount_untaxed",     DoubleType(),   True),
    StructField("amount_tax",         DoubleType(),   True),
    StructField("amount_total",       DoubleType(),   True),
    StructField("pricelist_id",       StringType(),   True),
    StructField("origin",             StringType(),   True),
    StructField("client_order_ref",   StringType(),   True),
    StructField("commitment_date",    TimestampType(), True),
    StructField("validity_date",      TimestampType(), True),
    StructField("created_at",         TimestampType(), True),
    StructField("updated_at",         TimestampType(), True),
    StructField("dwh_loaded_at",      TimestampType(), True),
    StructField("dwh_source_db",      StringType(),   True),
])

line_schema = StructType([
    StructField("sale_order_line_id",          IntegerType(),  False),
    StructField("sale_order_id",               IntegerType(),  True),
    StructField("sale_order_name",             StringType(),   True),
    StructField("company_id",                  IntegerType(),  True),
    StructField("company_name",                StringType(),   True),
    StructField("currency_id",                 IntegerType(),  True),
    StructField("currency_name",               StringType(),   True),
    StructField("partner_id",                  IntegerType(),  True),
    StructField("partner_name",                StringType(),   True),
    StructField("salesman_id",                 IntegerType(),  True),
    StructField("salesman_name",               StringType(),   True),
    StructField("product_id",                  IntegerType(),  True),
    StructField("product_name",                StringType(),   True),
    StructField("product_uom_id",              IntegerType(),  True),
    StructField("product_uom_name",            StringType(),   True),
    StructField("order_state",                 StringType(),   True),
    StructField("display_type",                StringType(),   True),
    StructField("virtual_id",                  IntegerType(),  True),
    StructField("linked_virtual_id",           IntegerType(),  True),
    StructField("qty_delivered_method",        StringType(),   True),
    StructField("invoice_status",              StringType(),   True),
    StructField("analytic_distribution",       StringType(),   True),
    StructField("extra_tax_data",              StringType(),   True),
    StructField("line_name",                   StringType(),   True),
    StructField("product_uom_qty",             DoubleType(),   True),
    StructField("price_unit",                  DoubleType(),   True),
    StructField("discount",                    DoubleType(),   True),
    StructField("price_subtotal",              DoubleType(),   True),
    StructField("price_total",                 DoubleType(),   True),
    StructField("price_reduce_taxexcl",        DoubleType(),   True),
    StructField("price_reduce_taxinc",         DoubleType(),   True),
    StructField("qty_delivered",               DoubleType(),   True),
    StructField("qty_invoiced",                DoubleType(),   True),
    StructField("qty_to_invoice",              DoubleType(),   True),
    StructField("untaxed_amount_invoiced",     DoubleType(),   True),
    StructField("untaxed_amount_to_invoice",   DoubleType(),   True),
    StructField("is_downpayment",              StringType(),   True),
    StructField("is_expense",                  StringType(),   True),
    StructField("collapse_prices",             StringType(),   True),
    StructField("collapse_composition",        StringType(),   True),
    StructField("created_at",                  TimestampType(), True),
    StructField("updated_at",                  TimestampType(), True),
    StructField("technical_price_unit",        DoubleType(),   True),
    StructField("price_tax",                   DoubleType(),   True),
    StructField("customer_lead",               DoubleType(),   True),
    StructField("is_optional",                 StringType(),   True),
    StructField("dwh_loaded_at",               TimestampType(), True),
    StructField("dwh_source_db",               StringType(),   True),
])

# Build name→index maps from the same lists used to build tuples
O = {name: i for i, name in enumerate(ORDER_COLS)}
L = {name: i for i, name in enumerate(LINE_COLS)}

# ── 6. Generate batch ─────────────────────────────────────────────────────────
rng = random.Random(SEED_BASE + run_number)
now = datetime.now(timezone.utc).replace(tzinfo=None)

pid_list     = sorted(products_map.keys())
partner_list = sorted(k for k in partners_map if 51 <= k <= 250)
user_list    = sorted(users_map.keys())

order_rows = []
line_rows  = []

for order_id in range(start_order_id, end_order_id):

    created    = _rdate(rng)
    partner_id = rng.choice(partner_list)
    state      = _order_state(rng)
    inv_status = _invoice_status(state, rng)
    order_name = f"S{order_id:06d}"
    user_id    = rng.choice(user_list)
    tid, tname = rng.choice(SALE_TEAMS)
    cid, cname = rng.choice(COMPANIES)
    cur_id, cur_name = CURRENCIES[0]

    total_untaxed = 0.0
    total_tax     = 0.0

    for _ in range(rng.randint(1, 3)):
        pid               = rng.choice(pid_list)
        list_price, pname = products_map[pid]
        qty               = float(rng.randint(1, 5))
        discount          = float(rng.choice([0, 5, 10, 15]))

        price_after_disc     = list_price * (1 - discount / 100)
        subtotal             = round(qty * price_after_disc, 2)
        price_tax_amt        = round(subtotal * TAX_RATE, 2)
        price_total          = round(subtotal + price_tax_amt, 2)
        price_reduce_taxexcl = round(price_after_disc, 2)
        price_reduce_taxinc  = round(price_after_disc * (1 + TAX_RATE), 2)

        qty_delivered  = (qty if state == "done"
                          else float(rng.randint(0, int(qty))) if state == "sale"
                          else 0.0)
        qty_invoiced   = qty if inv_status == "invoiced" else 0.0
        qty_to_inv     = max(qty_delivered - qty_invoiced, 0.0)
        untaxed_inv    = round(qty_invoiced * price_after_disc, 2)
        untaxed_to_inv = round(qty_to_inv   * price_after_disc, 2)

        total_untaxed += subtotal
        total_tax     += price_tax_amt

        line_dict = {
            "sale_order_line_id":          line_id,
            "sale_order_id":               order_id,
            "sale_order_name":             order_name,
            "company_id":                  cid,
            "company_name":                cname,
            "currency_id":                 cur_id,
            "currency_name":               cur_name,
            "partner_id":                  partner_id,
            "partner_name":                partners_map[partner_id],
            "salesman_id":                 user_id,
            "salesman_name":               users_map[user_id],
            "product_id":                  pid,
            "product_name":                pname,
            "product_uom_id":              1,
            "product_uom_name":            "Unit(s)",
            "order_state":                 state,
            "display_type":                None,
            "virtual_id":                  None,
            "linked_virtual_id":           None,
            "qty_delivered_method":        "manual",
            "invoice_status":              inv_status,
            "analytic_distribution":       None,
            "extra_tax_data":              None,
            "line_name":                   pname,
            "product_uom_qty":             qty,
            "price_unit":                  list_price,
            "discount":                    discount,
            "price_subtotal":              subtotal,
            "price_total":                 price_total,
            "price_reduce_taxexcl":        price_reduce_taxexcl,
            "price_reduce_taxinc":         price_reduce_taxinc,
            "qty_delivered":               qty_delivered,
            "qty_invoiced":                qty_invoiced,
            "qty_to_invoice":              qty_to_inv,
            "untaxed_amount_invoiced":     untaxed_inv,
            "untaxed_amount_to_invoice":   untaxed_to_inv,
            "is_downpayment":              "False",
            "is_expense":                  "False",
            "collapse_prices":             "False",
            "collapse_composition":        "False",
            "created_at":                  created,
            "updated_at":                  created,
            "technical_price_unit":        price_after_disc,
            "price_tax":                   price_tax_amt,
            "customer_lead":               0.0,
            "is_optional":                 "False",
            "dwh_loaded_at":               now,
            "dwh_source_db":               SOURCE_DB,
        }
        line_rows.append(tuple(line_dict[c] for c in LINE_COLS))
        line_id += 1

    amount_total = round(total_untaxed + total_tax, 2)

    order_dict = {
        "sale_order_id":    order_id,
        "partner_id":       partner_id,
        "partner_name":     partners_map[partner_id],
        "team_id":          tid,
        "team_name":        tname,
        "company_id":       cid,
        "company_name":     cname,
        "user_id":          user_id,
        "user_name":        users_map[user_id],
        "currency_id":      cur_id,
        "currency_name":    cur_name,
        "date_order":       created,
        "order_state":      state,
        "amount_untaxed":   round(total_untaxed, 2),
        "amount_tax":       round(total_tax, 2),
        "amount_total":     amount_total,
        "pricelist_id":     None,
        "origin":           None,
        "client_order_ref": None,
        "commitment_date":  None,
        "validity_date":    None,
        "created_at":       created,
        "updated_at":       created,
        "dwh_loaded_at":    now,
        "dwh_source_db":    SOURCE_DB,
    }
    order_rows.append(tuple(order_dict[c] for c in ORDER_COLS))

# ── 7. Pre-flight assertions ───────────────────────────────────────────────────
def _preflight():
    oids = [r[O["sale_order_id"]]      for r in order_rows]
    lids = [r[L["sale_order_line_id"]] for r in line_rows]

    assert len(oids) == BATCH_SIZE,     f"❌ Expected {BATCH_SIZE} orders, got {len(oids)}"
    assert len(oids) == len(set(oids)), "❌ Duplicate sale_order_id in this batch"
    assert len(lids) == len(set(lids)), "❌ Duplicate sale_order_line_id in this batch"
    assert min(oids) == start_order_id, f"❌ First order_id should be {start_order_id}"
    assert max(oids) == end_order_id-1, f"❌ Last order_id should be {end_order_id-1}"

    for o in order_rows:
        assert o[O["order_state"]] in {"draft","sent","sale","done","cancel"}
        assert 51 <= o[O["partner_id"]] <= 250
        assert round(o[O["amount_untaxed"]] + o[O["amount_tax"]], 2) == o[O["amount_total"]], \
            f"❌ Total mismatch on order {o[O['sale_order_id']]}"

    for l in line_rows:
        # [FIX 2 assertion] invoice_status must be one of 3 valid values only
        assert l[L["invoice_status"]] in {"to invoice","invoiced","no"}, \
            f"❌ Bad invoice_status: {l[L['invoice_status']]}"
        # cancel orders must never have invoice_status = invoiced
        if l[L["order_state"]] == "cancel":
            assert l[L["invoice_status"]] == "no", \
                f"❌ cancel order has invoice_status={l[L['invoice_status']]} on line {l[L['sale_order_line_id']]}"
        assert 1 <= l[L["product_id"]] <= 100, \
            f"❌ product_id out of range: {l[L['product_id']]}"
        assert l[L["discount"]] in {0.0, 5.0, 10.0, 15.0}, \
            f"❌ Bad discount: {l[L['discount']]}"

        expected_tax = round(l[L["price_subtotal"]] * TAX_RATE, 2)
        actual_tax   = l[L["price_tax"]]
        assert abs(actual_tax - expected_tax) < 0.02, \
            f"❌ Tax mismatch on line {l[L['sale_order_line_id']]}: " \
            f"got {actual_tax}, expected {expected_tax}"

    print(f"✅  Pre-flight passed — {len(oids)} orders, {len(lids)} lines")

_preflight()

# ── 8. Write to silver ────────────────────────────────────────────────────────
df_orders = spark.createDataFrame(order_rows, schema=order_schema)
df_lines  = spark.createDataFrame(line_rows,  schema=line_schema)

def append_silver(df, table: str) -> None:
    """
    Appends df to <table>.  Safe because each run holds a
    non-overlapping PK block derived from MAX() — existing rows are never touched.
    """
    if not spark.catalog.tableExists(table):
        (df.write
           .format("delta")
           .mode("errorifexists")
           .option("overwriteSchema", "false")
           .saveAsTable(table))
        print(f"  [created]  {table}  →  {df.count()} rows")
    else:
        (df.write
           .format("delta")
           .mode("append")
           .saveAsTable(table))
        print(f"  [appended] {table}  →  {df.count()} rows")

print("\nWriting silver layer …")
append_silver(df_orders, ORDER_TABLE)
append_silver(df_lines,  LINE_TABLE)

# ── 9. Update state table AFTER successful write ──────────────────────────────
if state_row:
    spark.sql(f"""
        UPDATE {STATE_TABLE}
        SET    run_number = {run_number},
               updated_at = current_timestamp()
        WHERE  table_name = 'sale_orders'
    """)
else:
    spark.sql(f"""
        INSERT INTO {STATE_TABLE}
        VALUES ('sale_orders', {run_number}, current_timestamp())
    """)

# ── 10. Summary ───────────────────────────────────────────────────────────────
total_orders = spark.sql(f"SELECT COUNT(*) FROM {ORDER_TABLE}").collect()[0][0]
total_lines  = spark.sql(f"SELECT COUNT(*) FROM {LINE_TABLE}").collect()[0][0]

print(f"""
✅  Sales simulation run #{run_number} complete.
   This run   : {len(order_rows):>6} orders  |  {len(line_rows):>6} lines
   Table total: {total_orders:>6} orders  |  {total_lines:>6} lines
   order_ids  : {start_order_id} → {end_order_id - 1}
""")